# 🎙️ Colab TTS Worker — OmniVoice
Kết nối server qua WebSocket, xử lý TTS task bằng [k2-fsa/OmniVoice](https://github.com/k2-fsa/OmniVoice).

In [ ]:
#@title 1. Cài đặt & Load Model
import os, sys, time

# Kiểm tra GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✅ GPU: {gpu_name}')
else:
    print('⚠️ KHÔNG có GPU! Worker sẽ chạy rất chậm trên CPU.')

# Cài đặt OmniVoice (Colab đã có sẵn torch, torchaudio)
# QUAN TRỌNG: --no-deps để tránh downgrade PyTorch sang CPU-only
print('📦 Đang cài đặt OmniVoice...')
!pip install -q --upgrade "transformers>=5.3"
!pip install -q omnivoice --no-deps
!pip install -q soundfile pydub soxr websockets httpx

# Load model
print('🔄 Đang tải model OmniVoice...')
t0 = time.time()
from omnivoice import OmniVoice
import soundfile as sf

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map=DEVICE,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
load_ms = round((time.time() - t0) * 1000, 1)
print(f'✅ Model loaded trên {DEVICE} trong {load_ms:.0f}ms')

In [ ]:
#@title 2. WebSocket Worker — Kết nối Server & Xử lý Task
import asyncio
import json
import io
import os
import time
import hashlib
import httpx
import websockets
import torch
import soundfile as sf

# ── CONFIG ────────────────────────────────────────────────
SERVER_URL = ""  #@param {type:"string"}
EMAIL = ""       #@param {type:"string"}

SAMPLE_RATE = 24000  # OmniVoice output 24kHz
REF_CACHE_DIR = "/tmp/omnivoice_refs"
os.makedirs(REF_CACHE_DIR, exist_ok=True)

# ── Chuẩn hóa URL (bắt lỗi sớm, không để loop với URI sai) ──
SERVER_URL = (SERVER_URL or "").strip().rstrip("/")
if not SERVER_URL or not (SERVER_URL.startswith("http://") or SERVER_URL.startswith("https://")):
    raise RuntimeError(
        f"SERVER_URL không hợp lệ: '{SERVER_URL}'\n"
        "Hãy điền URL Cloudflare (https://xxx.trycloudflare.com) vào form @param ở trên rồi chạy lại cell."
    )
WS_URL = SERVER_URL.replace("https://", "wss://").replace("http://", "ws://") + "/ws/worker"
print(f"SERVER_URL : {SERVER_URL}")
print(f"WS_URL     : {WS_URL}")
print(f"EMAIL      : {EMAIL}")


def run_tts(text: str, ref_audio_path: str, ref_text: str = "", language: str = None) -> bytes:
    kwargs = dict(text=text, ref_audio=ref_audio_path)
    if language:
        kwargs["language"] = language
    if ref_text:
        kwargs["ref_text"] = ref_text
    try:
        audios = model.generate(**kwargs)
    except TypeError as exc:
        if "language" not in kwargs:
            raise
        print(f"[warn] Model không nhận language={language}, fallback auto: {exc}")
        kwargs.pop("language", None)
        audios = model.generate(**kwargs)
    buf = io.BytesIO()
    sf.write(buf, audios[0], SAMPLE_RATE, format="WAV")
    return buf.getvalue()


async def download_ref_audio(http_client: httpx.AsyncClient, voice_url: str) -> str:
    full_url = voice_url if voice_url.startswith("http") else f"{SERVER_URL}{voice_url}"
    url_hash = hashlib.md5(full_url.encode()).hexdigest()[:12]
    cached = os.path.join(REF_CACHE_DIR, f"ref_{url_hash}.wav")
    if os.path.exists(cached):
        print(f"  [cache] Ref audio: {url_hash}")
        return cached
    print(f"  [download] {full_url}")
    resp = await http_client.get(full_url, timeout=30)
    resp.raise_for_status()
    with open(cached, "wb") as f:
        f.write(resp.content)
    return cached


async def worker_loop():
    retry_delay = 5
    max_delay = 60

    async with httpx.AsyncClient() as http_client:
        while True:
            try:
                print(f"[ws] Đang kết nối: {WS_URL} ...")
                async with websockets.connect(
                    WS_URL,
                    open_timeout=30,
                    close_timeout=10,
                    ping_interval=30,
                    ping_timeout=20,
                ) as ws:
                    retry_delay = 5
                    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
                    await ws.send(json.dumps({"action": "register", "email": EMAIL, "gpu": gpu_name}))
                    print(f"[ws] Đã đăng ký: {EMAIL} (GPU: {gpu_name})")
                    await ws.send(json.dumps({"action": "status", "status": "IDLE"}))

                    while True:
                        raw = await ws.recv()
                        data = json.loads(raw)
                        action = data.get("action")

                        if action == "run_tts":
                            task_id = data["task_id"]
                            text = data["text"]
                            voice_url = data["voice_api_url"]
                            language = data.get("language")
                            print(f"[task] {task_id} | {text[:60]}{'...' if len(text)>60 else ''}")
                            try:
                                await ws.send(json.dumps({"action": "status", "status": "BUSY"}))
                                ref_path = await download_ref_audio(http_client, voice_url)
                                t0 = time.time()
                                result_audio = run_tts(text, ref_path, language=language)
                                tts_ms = round((time.time() - t0) * 1000, 1)
                                upload_url = f"{SERVER_URL}/api/tasks/{task_id}/complete"
                                upload_resp = await http_client.post(
                                    upload_url,
                                    files={"audio": ("result.wav", result_audio, "audio/wav")},
                                    timeout=120,
                                )
                                upload_resp.raise_for_status()
                                await ws.send(json.dumps({"action": "task_completed", "task_id": task_id}))
                                dur = len(result_audio) / (SAMPLE_RATE * 2)
                                print(f"[ok] {task_id} TTS={tts_ms}ms audio~{dur:.1f}s")
                            except Exception as e:
                                print(f"[fail] {task_id}: {e}")
                                try:
                                    await ws.send(json.dumps({"action": "task_failed", "task_id": task_id, "error": str(e)}))
                                except Exception:
                                    pass
                            finally:
                                try:
                                    await ws.send(json.dumps({"action": "status", "status": "IDLE"}))
                                except Exception:
                                    pass

                        elif action == "ping":
                            await ws.send(json.dumps({"action": "pong"}))
                        elif action == "shutdown":
                            print("[ws] Server yêu cầu shutdown.")
                            return

            except (websockets.ConnectionClosed, OSError) as e:
                print(f"[ws] Mất kết nối: {e}. Thử lại sau {retry_delay}s...")
                await asyncio.sleep(retry_delay)
                retry_delay = min(retry_delay * 2, max_delay)
            except Exception as e:
                print(f"[ws] Lỗi: {e}. Thử lại sau {retry_delay}s...")
                await asyncio.sleep(retry_delay)
                retry_delay = min(retry_delay * 2, max_delay)


print("[worker] Sẵn sàng. Đang khởi chạy...")
await worker_loop()
